In [ ]:
import requests
from selenium import webdriver
import time
from bs4 import BeautifulSoup

In [ ]:
url = "https://www.printables.com/search/models?q=cat"

driver = webdriver.Chrome()
driver.maximize_window()
driver.get(url)

# Wait for page to load
time.sleep(3)

# Remove cookie popup
driver.execute_script("""
    document.querySelectorAll('[class*="cookie"], [id*="cookie"]').forEach(el => el.remove());
""")
print("✓ Cookie popup removed")

# Scroll to load all content
last_height = 0
scroll_count = 0

print("Starting scroll...")
while scroll_count < 20:
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
    time.sleep(2)
    
    new_height = driver.execute_script("return document.body.scrollHeight")
    scroll_count += 1
    
    print(f"Scroll {scroll_count}: {new_height} (prev={last_height})")
    
    if new_height == last_height:
        print("✓ Reached bottom")
        break
        
    last_height = new_height

print(f"Done! {scroll_count} scrolls")

✓ Overlays removed
Starting scroll...
Starting scroll...
Scroll 1: height=7446 (prev=0)
Scroll 1: height=7446 (prev=0)
Scroll 2: height=10839 (prev=7446)
Scroll 2: height=10839 (prev=7446)
Scroll 3: height=14232 (prev=10839)
Scroll 3: height=14232 (prev=10839)
Scroll 4: height=18002 (prev=14232)
Scroll 4: height=18002 (prev=14232)
Scroll 5: height=21395 (prev=18002)
Scroll 5: height=21395 (prev=18002)
Scroll 6: height=24788 (prev=21395)
Scroll 6: height=24788 (prev=21395)
Scroll 7: height=28181 (prev=24788)
Scroll 7: height=28181 (prev=24788)
Scroll 8: height=31951 (prev=28181)
Scroll 8: height=31951 (prev=28181)
Scroll 9: height=35344 (prev=31951)
Scroll 9: height=35344 (prev=31951)
Scroll 10: height=38737 (prev=35344)
Scroll 10: height=38737 (prev=35344)
Scroll 11: height=42130 (prev=38737)
Scroll 11: height=42130 (prev=38737)
Scroll 12: height=45900 (prev=42130)
Scroll 12: height=45900 (prev=42130)
Scroll 13: height=49293 (prev=45900)
Scroll 13: height=49293 (prev=45900)
Scroll 14: 

In [2]:
headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }

In [3]:
def parse_article(html):
    articles = html.find_all("article", {"data-testid": "model"})
    result = []
    
    for article in articles:
        # --- title and model link ---
        title_tag = article.select_one("h5 a.h")
        title = title_tag.get_text(strip=True) if title_tag else None
        model_link = "https://www.printables.com" + title_tag["href"] if title_tag else None

        # --- author name and link ---
        author_tag = article.select_one("a.username")
        author = author_tag.get_text(strip=True) if author_tag else None
        author_link = "https://www.printables.com" + author_tag["href"] if author_tag else None

        # --- model image ---
        img_tag = article.select_one("a.card-image img")
        image_url = img_tag["src"] if img_tag else None
        if image_url and image_url.startswith("data:image"):  # skip placeholder
            second_img = article.select_one("picture.image-inside img")
            if second_img:
                image_url = second_img["src"]

        # --- stats from stats-bar ---
        stats_bar = article.select_one("div.stats-bar")
        like_count = None
        download_count = None
        
        if stats_bar:
            like_span = stats_bar.select_one("span[data-testid='like-count']")
            like_count = like_span.get_text(strip=True) if like_span else None
            
            download_icon = stats_bar.select("div.small-icon i.fa-arrow-down-to-line")
            if download_icon:
                download_span = download_icon[0].find_next_sibling("span")
                download_count = download_span.get_text(strip=True) if download_span else None

        result.append({
            "title": title,
            "image": image_url,
            "model_link": model_link,
            "author": author,
            "author_link": author_link,
            "like_count": like_count,
            "download_count": download_count,
        })
    return result

In [4]:
def scrape_printables(query="dragon", pages=3, headers={}):
    results = []
    for page in range(1, pages + 1):
        url = f"https://www.printables.com/search/models?ctx=models&q={query}&page={page}"

        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")
        models = parse_article(soup)

        print(f"Page {page}: found {len(models)} models")
        results.extend(models)

    return results

In [5]:
all_models = scrape_printables("car", pages=5, headers=headers)
print(f"Total models scraped: {len(all_models)}")

Page 1: found 36 models
Page 2: found 36 models
Page 3: found 36 models
Page 4: found 36 models
Page 5: found 36 models
Total models scraped: 180
